In [1]:
import pandas as pd
import joblib

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# ======================
# SENSOR MODEL
# ======================

sensor_df = pd.read_csv("/content/CNC_WashingMachine_Anomaly_Dataset.csv")

X_sensor = sensor_df.drop("Status", axis=1)
y_sensor = sensor_df["Status"]

sensor_encoder = LabelEncoder()
y_sensor = sensor_encoder.fit_transform(y_sensor)

cat_cols = ["Machine_Type"]

num_cols = [
    "Temperature_C",
    "Vibration_mm_s",
    "RPM",
    "Pressure_bar",
    "Humidity_percent",
    "Power_kW"
]

preprocessor = ColumnTransformer([
    (
        "cat",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]),
        cat_cols
    ),
    (
        "num",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median"))
        ]),
        num_cols
    )
])

sensor_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        random_state=42
    ))
])

sensor_model.fit(X_sensor, y_sensor)

# ======================
# TEXT MODEL
# ======================

text_df = pd.read_csv("/content/text_dataset.csv")

X_text = text_df["text"]
y_text = text_df["label"]

text_encoder = LabelEncoder()
y_text = text_encoder.fit_transform(y_text)

text_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=5000,
        ngram_range=(1,2)
    )),
    ("classifier", LogisticRegression(
        max_iter=1000
    ))
])

text_model.fit(X_text, y_text)

# ======================
# COMBINED MODEL
# ======================

combined_model = {
    "sensor_model": sensor_model,
    "text_model": text_model,
    "sensor_encoder": sensor_encoder,
    "text_encoder": text_encoder
}

joblib.dump(
    combined_model,
    "machine_anomaly_model.pkl"
)

print("Single PKL file saved.")

Single PKL file saved.
